# 03: LangSmith, Observe, Evaluate, Improve

**12 components, one section each**

This notebook follows the [LangSmith SDK map](assets/langsmith_api_landscape.png) component by component. Each section covers one component and runs its most useful methods as examples. Anything else in that component is listed at the end of its section so you know it exists.

| # | Component | Examples we run |
|---|---|---|
| 1 | Configure | `LANGSMITH_TRACING=true`, `LANGSMITH_API_KEY`, `LANGSMITH_PROJECT` |
| 2 | Tracing | `@traceable`, `trace()` context manager |
| 3 | Wrappers & interop | `wrap_openai()` |
| 4 | Projects & runs | `Client()`, `list_runs(filter=)` |
| 5 | Datasets | `create_dataset()`, `create_examples()` |
| 6 | Evaluation | `evaluate(target, data=...)`, `(inputs, outputs, reference_outputs)` |
| 7 | Judges & metrics | `create_llm_as_judge()` |
| 8 | Testing | `@pytest.mark.langsmith` |
| 9 | Feedback | `create_feedback(run_id, key, score)` |
| 10 | Prompt Hub | `push_prompt()`, `pull_prompt('name:tag')` |
| 11 | Human review | `create_annotation_queue()` |
| 12 | Monitor (UI) | online evaluators |

LangSmith works with *any* framework, even raw OpenAI calls. Keep the LangSmith UI open in a second window and flip to it after each section.

## Setup (run once)

A LangSmith key is **required** for this notebook (the free tier is enough): https://smith.langchain.com -> Settings -> API Keys.

Everything runs on `gpt-4o-mini`, so it is cheap (a full run of this notebook costs a few cents). Swap `MODEL` for any provider you like.

In [1]:
%pip install -q -U langsmith langchain langchain-openai openai pydantic pandas pytest openevals

Note: you may need to restart the kernel to use updated packages.


  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.62.1 requires numpy<2.4,>=1.22, but you have numpy 2.5.4 which is incompatible.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 2.5.4 which is incompatible.


In [7]:
import os, getpass

def set_env(var: str):
    """Ask for a secret only if it is not already in the environment."""
    if not os.environ.get(var):
        val = getpass.getpass(f"{var} (Enter to skip): ")
        if val:
            os.environ[var] = val

set_env("OPENAI_API_KEY")
set_env("LANGSMITH_API_KEY")

if os.environ.get("LANGSMITH_API_KEY"):
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ["LANGSMITH_PROJECT"] = "ai-coding-agent-meetup"
    print("LangSmith tracing ON -> project: ai-coding-agent-meetup")
else:
    print("LangSmith tracing OFF")

MODEL = "openai:gpt-4o-mini"   # provider:model - change freely

LangSmith tracing ON -> project: ai-coding-agent-meetup


In [8]:
assert os.environ.get("LANGSMITH_API_KEY"), "This notebook needs a LangSmith API key (free tier is fine)."
from langsmith import Client
client = Client()
print("Connected to LangSmith. Project:", os.environ["LANGSMITH_PROJECT"])

Connected to LangSmith. Project: ai-coding-agent-meetup


## 1. Configure
Tracing is switched on with three environment variables, and the setup cell above already set them:

- `LANGSMITH_TRACING=true` turns tracing on
- `LANGSMITH_API_KEY` says who you are
- `LANGSMITH_PROJECT` chooses where runs are grouped

With those set, every LangChain / LangGraph run is recorded with no code changes. Let's make one.

**Also worth knowing:** `LANGSMITH_ENDPOINT`, `LANGSMITH_WORKSPACE_ID`, `LANGSMITH_HIDE_INPUTS`, tracing sample rate

In [9]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = init_chat_model(MODEL, temperature=0)
fun_facts = ChatPromptTemplate.from_template("Give me 3 fun facts about {topic}.") | llm | StrOutputParser()

print(fun_facts.invoke({"topic": "octopuses"}))

Sure! Here are three fun facts about octopuses:

1. **Three Hearts and Blue Blood**: Octopuses have three hearts—two pump blood to the gills, where it gets oxygenated, and the third pumps it to the rest of the body. Their blood is blue due to the presence of hemocyanin, a copper-based molecule that is more efficient than hemoglobin in cold, low-oxygen environments.

2. **Incredible Intelligence**: Octopuses are known for their remarkable intelligence and problem-solving abilities. They can navigate mazes, open jars to access food, and even use tools, such as coconut shells and rocks, for shelter or defense.

3. **Master of Disguise**: Octopuses have specialized skin cells called chromatophores that allow them to change color and texture almost instantaneously. This ability helps them blend into their surroundings for camouflage, communicate with other octopuses, and even express emotions. Some species can mimic the appearance of other marine animals as a form of defense against predato

Switch to the LangSmith UI → *Tracing* → project `meetup-langsmith`. You should see one run. Click it and look at, in this order:

1. **Input / Output** of the whole chain
2. **Child runs**: `ChatPromptTemplate` → `ChatOpenAI` → `StrOutputParser`
3. The **LLM run**: exact messages sent, model params, **tokens, cost, latency**
4. **Waterfall / timeline** view: where did the time go?

## 2. Tracing
Not everything is a LangChain Runnable. A **trace** is a tree of **runs** (spans), and you can add your own:

- `@traceable` traces a whole Python function. Calls nest automatically, so child runs attach to the parent.
- `trace()` is a context manager that traces just a block of code.

**Also worth knowing:** `run_type=`, `name= · tags= · metadata=`, `tracing_context()`, `get_current_run_tree()`, `langsmith_extra=`, `process_inputs/outputs`

In [10]:
from langsmith import traceable

@traceable(name="clean_text", run_type="tool")
def clean_text(text: str) -> str:
    return " ".join(text.split()).strip()

summary_chain = ChatPromptTemplate.from_template("Summarise in one sentence:\n{text}") | llm | StrOutputParser()

@traceable(name="summarize_pipeline", run_type="chain")
def summarize(text: str) -> str:
    cleaned = clean_text(text)                       # child run 1 (plain Python)
    return summary_chain.invoke({"text": cleaned})   # child run 2 (LangChain, nested automatically)

print(summarize("   LangSmith   records   every   step   of   your   LLM   app,\n   so   you   can   debug   it.   "))

LangSmith tracks all actions in your LLM application, enabling effective debugging.


### `trace()` context manager
Use it when you do not own the function (or only want to trace a few lines). You can also give the run a name, **tags** (`prod`, `experiment-3`) and **metadata** (`user_id`, `app_version`), so you can filter by them in the UI.

In [11]:
from langsmith import trace

with trace(
    name="fun-facts-demo",
    run_type="chain",
    inputs={"topic": "black holes"},
    tags=["meetup", "demo"],
    metadata={"user_id": "u-42", "app_version": "1.0.3", "env": "demo"},
) as run:
    answer = fun_facts.invoke({"topic": "black holes"})
    run.end(outputs={"answer": answer})

print("done - filter by tag 'meetup' or metadata user_id=u-42 in the UI")

done - filter by tag 'meetup' or metadata user_id=u-42 in the UI


## 3. Wrappers & interop
No LangChain at all? Wrap the client. `wrap_openai()` returns the same OpenAI client, but every call is traced. LangChain and LangGraph runs are traced automatically with no wrapper.

**Also worth knowing:** `wrap_anthropic()`, LangChain auto-trace, LangGraph auto-trace, OpenTelemetry, `RunTree` (manual)

In [6]:
from openai import OpenAI
from langsmith.wrappers import wrap_openai

oai = wrap_openai(OpenAI())   # same client, now traced

r = oai.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Write a haiku about debugging."}],
)
print(r.choices[0].message.content)

Lines of code entwined,  
Errors hide in shadows deep,  
Logic's dance revealed.


## 4. Projects & runs
Everything in the UI is available through the API. `Client()` is the entry point (we created it in the setup check above), and `list_runs(filter=)` queries runs with the same filter language as the UI. Use it for custom dashboards, cost reports, or nightly jobs that harvest bad runs into a dataset.

**Also worth knowing:** `read_run()`, `share_run()`, `get_run_url()`, `create_project()`, `list_threads()`, `flush()`

In [12]:
import pandas as pd

client.flush()   # traces upload in the background; make sure the earlier runs have arrived

runs = list(client.list_runs(
    project_name=os.environ["LANGSMITH_PROJECT"],
    is_root=True,
    filter='has(tags, "meetup")',     # same filter language as the UI
    limit=50,
))

rows = [
    {
        "name": r.name,
        "latency_s": (r.end_time - r.start_time).total_seconds(),
        "total_tokens": r.total_tokens,
        "error": bool(r.error),
        "tags": ",".join(r.tags or []),
    }
    for r in runs if r.end_time and r.start_time
]

df = pd.DataFrame(rows)
display(df.head(10))
if not df.empty:
    print("Avg latency (s):", round(df["latency_s"].mean(), 2),
          "| total tokens:", int(df["total_tokens"].fillna(0).sum()),
          "| error rate:", round(df["error"].mean(), 3))

C:\Users\91801\AppData\Local\Temp\ipykernel_11944\991450637.py:5: LangSmithDeprecationWarning: list_runs() is deprecated and will be removed after Jan 31, 2027. Use client.runs.query() instead.See https://docs.langchain.com/langsmith/smithdb-sdk-migration-query-runs#runs-query for the migration guide.
  runs = list(client.list_runs(


""


## 5. Datasets
A **dataset** is a versioned set of **examples** (`inputs` + optional reference `outputs`). It is your test set.

- `create_dataset()` makes the dataset
- `create_examples()` fills it

You can also build datasets from a CSV upload, or by **saving production traces** with one click. The best dataset is made of real failures you saw in traces: add every bug to it and it can never silently regress.

**Also worth knowing:** `list_examples()`, `update_example()`, `create_example_from_run()`, `upload_csv()`, dataset splits, versions & tags

In [13]:
DATASET = "meetup-qa-demo"

examples = [
    ("What is 17 * 23?",                      "391"),
    ("What is the capital of Australia?",     "Canberra"),
    ("Who wrote the play Hamlet?",            "Shakespeare"),
    ("What is the chemical symbol for gold?", "Au"),
    ("Which planet is the largest in our solar system?", "Jupiter"),
]

if not client.has_dataset(dataset_name=DATASET):
    ds = client.create_dataset(DATASET, description="Tiny Q&A set for the meetup demo")
    client.create_examples(
        dataset_id=ds.id,
        inputs=[{"question": q} for q, _ in examples],
        outputs=[{"answer": a} for _, a in examples],
    )
    print("Created dataset with", len(examples), "examples")
else:
    print("Dataset already exists - reusing it")

Created dataset with 5 examples


## 6. Evaluation
`evaluate(target, data=...)` runs your app on every example and scores each output. The result is an **experiment** you can open in the UI.

- **target**: a function taking the example `inputs` dict and returning an `outputs` dict
- **evaluator**: a function with the signature `(inputs, outputs, reference_outputs)` that returns a score. Ask only for the arguments you need.

Code evaluators are cheap and deterministic, so use them for facts.

**Also worth knowing:** `aevaluate()`, `evaluators=[fn]`, `summary_evaluators`, `experiment_prefix`, `num_repetitions`, `max_concurrency`, `upload_results`

In [ ]:
from langsmith import evaluate

def make_target(system_prompt: str, model: str = MODEL):
    model_ = init_chat_model(model, temperature=0)
    chain = ChatPromptTemplate.from_messages([("system", system_prompt), ("human", "{question}")]) | model_ | StrOutputParser()
    def target(inputs: dict) -> dict:
        return {"answer": chain.invoke({"question": inputs["question"]})}
    return target

# --- code evaluators: (inputs, outputs, reference_outputs), take only what you need ---
def contains_answer(outputs: dict, reference_outputs: dict) -> dict:
    ok = reference_outputs["answer"].lower() in outputs["answer"].lower()
    return {"key": "contains_answer", "score": int(ok)}

def is_brief(outputs: dict) -> dict:
    return {"key": "is_brief", "score": int(len(outputs["answer"].split()) <= 8)}

TERSE = "Answer with the shortest possible answer. No explanations."
res_terse = evaluate(
    make_target(TERSE),
    data=DATASET,
    evaluators=[contains_answer, is_brief],
    experiment_prefix="terse-prompt",
    max_concurrency=4,
)

### A second experiment, for comparison
Run the same dataset with a different prompt. Two experiments on one dataset is how you A/B test a prompt (or a model): change one thing, run, compare.

In [ ]:
VERBOSE = "You are a friendly tutor. Explain your answer step by step before giving it."

res_verbose = evaluate(
    make_target(VERBOSE),
    data=DATASET,
    evaluators=[contains_answer, is_brief],
    experiment_prefix="verbose-prompt",
    metadata={"prompt_version": "verbose-v1"},
    max_concurrency=4,
)

In the UI: *Datasets & Experiments* → `meetup-qa-demo` → tick `terse-prompt-...` and `verbose-prompt-...` → **Compare**. Look at:
- per-example scores and the average
- regressions / improvements highlighted per example
- side-by-side outputs
- latency, token and cost columns

This is the workflow: change prompt → run experiment → compare → ship only if scores *and* cost are OK.

## 7. Judges & metrics
For fuzzy criteria ("is it correct? helpful? polite?") use a second LLM as the grader. The `openevals` package provides `create_llm_as_judge()` with ready-made prompts such as `CORRECTNESS_PROMPT`, so you do not have to write the grading prompt yourself.

A judge returns `{key, score, comment}`. The comment (the reason) is what makes judge scores debuggable.

**Also worth knowing:** `CORRECTNESS_PROMPT`, comparative `evaluate((a, b))`, trajectory match, custom rubric, `{key, score, comment}`

In [ ]:
from openevals.llm import create_llm_as_judge
from openevals.prompts import CORRECTNESS_PROMPT

correctness_judge = create_llm_as_judge(
    prompt=CORRECTNESS_PROMPT,
    feedback_key="correctness",
    model="openai:gpt-4o-mini",
)

def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    return correctness_judge(
        inputs=inputs["question"],
        outputs=outputs["answer"],
        reference_outputs=reference_outputs["answer"],
    )

res_judged = evaluate(
    make_target(VERBOSE),
    data=DATASET,
    evaluators=[contains_answer, correctness],
    experiment_prefix="verbose-prompt-judged",
    max_concurrency=4,
)

## 8. Testing
`@pytest.mark.langsmith` turns LangSmith into your **CI quality gate**. Each test logs its inputs, outputs, reference outputs and feedback to an experiment, so a failing test comes with a full trace. Run it in GitHub Actions on every pull request.

We first write a small test file, then run it.

**Also worth knowing:** `langsmith.testing.log_inputs`, `log_outputs`, `log_reference_outputs`, `log_feedback`, CI regression gates

In [ ]:
%%writefile test_qa_langsmith.py
import pytest
from langsmith import testing as t
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

chain = (
    ChatPromptTemplate.from_messages([("system", "Answer with the shortest possible answer."), ("human", "{q}")])
    | init_chat_model("openai:gpt-4o-mini", temperature=0)
    | StrOutputParser()
)

CASES = [("What is 9 * 9?", "81"), ("Capital of France?", "Paris"), ("Chemical symbol for sodium?", "Na")]

@pytest.mark.langsmith
@pytest.mark.parametrize("question, expected", CASES)
def test_short_correct_answers(question, expected):
    t.log_inputs({"question": question})
    answer = chain.invoke({"q": question})
    t.log_outputs({"answer": answer})
    t.log_reference_outputs({"answer": expected})
    t.log_feedback(key="contains_answer", score=int(expected.lower() in answer.lower()))
    assert expected.lower() in answer.lower()

Now run it. The suite name goes in an environment variable, which we set from Python so the same cell works on Windows, macOS and Linux. In CI you would simply run `pytest` as a pipeline step.

In [ ]:
import subprocess, sys

env = {**os.environ, "LANGSMITH_TEST_SUITE": "meetup-ci-suite"}
result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "test_qa_langsmith.py"],
    env=env, capture_output=True, text=True,
)
print(result.stdout[-1500:])
print(result.stderr[-500:])

## 9. Feedback
Attach **human signals** (thumbs up/down, star rating, comments) to any run. In production you capture the `run_id` when you answer, then your UI's thumbs button calls `create_feedback(run_id, key, score)`.

In the UI: open that run → **Feedback** tab. You can also *filter runs by feedback* (e.g. "all thumbs-down from last week") and push them into a dataset.

**Also worth knowing:** `list_feedback()`, `create_presigned_feedback_token()`, feedback configs, thumbs up / down UI

In [ ]:
import uuid, time

@traceable(name="support_bot")
def support_bot(question: str) -> str:
    return (ChatPromptTemplate.from_template("You are a support agent. Answer briefly: {q}") | llm | StrOutputParser()).invoke({"q": question})

run_id = uuid.uuid4()
answer = support_bot("How do I reset my password?", langsmith_extra={"run_id": run_id})
print(answer)

time.sleep(5)   # traces upload in the background; the run must exist before feedback is attached
client.create_feedback(run_id, key="user_score", score=1, comment="Clear and helpful")
print("Feedback attached to run", run_id)

## 10. Prompt Hub
Keep prompts **outside** your code: version them, review them, roll back, and let non-engineers edit them in the Playground.

- `push_prompt(name, object=...)` saves a new version
- `pull_prompt('name:tag')` loads one at runtime, by name, commit or tag (such as `prod`)

> Pushing needs a workspace handle; if it fails in your account the cell prints a hint.

**Also worth knowing:** `list_prompts()`, commits, tags (prod / staging), Playground

In [ ]:
prompt_name = "meetup-terse-qa"
qa_prompt = ChatPromptTemplate.from_messages([("system", TERSE), ("human", "{question}")])

try:
    url = client.push_prompt(prompt_name, object=qa_prompt)
    print("Pushed:", url)
    pulled = client.pull_prompt(prompt_name)           # or f"{prompt_name}:prod" for a tagged version
    print((pulled | llm | StrOutputParser()).invoke({"question": "Capital of Japan?"}))
except Exception as e:
    print("Prompt Hub step skipped:", type(e).__name__, "-", str(e)[:200])
    print("-> Create a workspace handle in LangSmith settings, or push from the Playground.")

## 11. Human review
This is the step after automation: people check what the machines scored. An **annotation queue** collects runs for human reviewers, who label them in the UI. Good labelled runs can then be pushed into a dataset (section 5).

**Also worth knowing:** `add_runs_to_annotation_queue()`, rubrics, reviewers → dataset

In [ ]:
try:
    queue = client.create_annotation_queue(name="meetup-review", description="Runs a human should check")
    client.add_runs_to_annotation_queue(queue.id, run_ids=[run_id])   # the support_bot run from section 9
    print("Queued run", run_id, "in", queue.name)
except Exception as e:
    print("Annotation queue step skipped:", type(e).__name__, "-", str(e)[:200])

## 12. Monitor (UI)
No code here: this section lives in the LangSmith web app (smith.langchain.com). The first feature to know is **online evaluators**: they score a share of your *live* traces automatically (toxicity, helpfulness, hallucination), so you notice problems without waiting for a user to complain.

| Feature | Where | What it gives you |
|---|---|---|
| **Online evaluators** | Project → *Evaluators* | Auto-score a % of live traces |
| Dashboards | Project → *Monitor* | Latency p50/p99, error rate, token usage |
| Rules & automations | Project → *Rules* | "If feedback = thumbs-down → add to dataset / send to annotation queue / webhook" |
| Alerts | Project → *Monitor* | Notify Slack or PagerDuty when a metric crosses a threshold |
| Insights clustering | Project → *Insights* | Group similar runs to spot common failure patterns |
| Cost & latency | Project → *Monitor* | Spend and speed per model and per prompt version |

The loop that matters: **Trace → find failures → add to dataset → fix prompt → run experiment → compare → deploy → monitor.** LangSmith is every arrow in that loop.

---
## Recap

1. Tracing is 3 env vars; `@traceable`, `trace()` and `wrap_openai()` cover everything else.
2. `Client()` + `list_runs(filter=)` give you the same data as the UI, in code.
3. **Datasets + evaluators + experiments** = unit tests for LLM apps. Use code evaluators for facts and `create_llm_as_judge()` for fuzzy quality.
4. `@pytest.mark.langsmith` puts those checks in CI.
5. `create_feedback()`, annotation queues and online evaluators bring **people and live traffic** into the loop.
6. Prompt Hub (`push_prompt()` / `pull_prompt()`) keeps prompts versioned outside your code.

**Next:** `04_langgraph.ipynb` → build stateful agents, then (optionally) revisit LangSmith to trace them.

### Optional exercises
- Add a third prompt variant ("answer in French") and watch `contains_answer` drop.
- Write an evaluator that penalises answers longer than 20 tokens.
- Add a rule in the UI that routes thumbs-down runs to an annotation queue.